<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/tadp_governance_cifar_10clients_5runs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow scikit-learn matplotlib pandas numpy psutil

In [ ]:
# ======================================================================================
# TADP — CIFAR-10 CROSS-DATASET GENERALIZATION (CPU-OPTIMIZED, 3 SCENARIOS)
# Q1-focused design: Full FedAvg vs TADP-VR vs matched Random-K
# ======================================================================================
# Scientific question:
#   Does the revised TADP governance mechanism generalize to an image-domain FL task
#   while preserving predictive utility close to full-participation FedAvg?
#
# Scenarios:
#   1) Full FedAvg  : all 10 clients
#   2) TADP-VR      : fixed governance-admitted cohort
#   3) Random-K     : fixed random cohort per run, same K / rounds / optimizer-step
#                     budget / initialization as TADP-VR
#
# Leakage safeguards:
#   • official CIFAR-10 TEST is held out from partitioning, preprocessing fitting,
#     TADP scoring, client admission, and training;
#   • only official TRAIN is partitioned among clients;
#   • RGB normalization is estimated from aggregated local TRAIN-only sufficient stats;
#   • documentary evidence is fixed before ML training and frozen across all 5 runs;
#   • CIFAR DQ is machine-measured from TRAIN only and frozen before ML training;
#   • target/class proportions are diagnostic only and never an admission criterion;
#   • distribution-quality checks are class-conditional to avoid penalizing the
#     intentionally non-IID class mixture by construction.
#
# NOTE: 10 FL rounds × 1 local epoch is a fixed comparison budget. A TRAIN-derived
# validation split is used for learning curves/stabilization diagnostics; the official
# CIFAR-10 TEST split is never used for model/configuration selection.
# ======================================================================================

from __future__ import annotations

import os, gc, json, math, time, random, hashlib, zipfile, platform
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from scipy.stats import t as student_t

# ======================================================================================
# 1. FIXED EXPERIMENT CONFIGURATION
# ======================================================================================

EXPERIMENT_VERSION = "TADP-CIFAR10 v3.0 — CPU-optimized 3-scenario generalization"
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUT = ROOT / "TADP_CIFAR10_3SCENARIO_Q1_v3"
TABLE_DIR, LEDGER_DIR, FIG_DIR = OUT / "tables", OUT / "ledgers", OUT / "figures"
for d in (OUT, TABLE_DIR, LEDGER_DIR, FIG_DIR): d.mkdir(parents=True, exist_ok=True)

NUM_CLIENTS, N_RUNS, DIRICHLET_ALPHA = 10, 5, 1.0
# Ten communication rounds gives every selected client repeated exposure to its local
# partition while keeping the experiment practical on CPU. This is a fixed, predeclared
# budget; validation curves are reported to show whether learning has stabilized.
NUM_ROUNDS_FL, LOCAL_EPOCHS, BATCH_SIZE, LR_LOCAL = 10, 1, 128, 1e-3
VALIDATION_FRACTION = 0.10
PARTITION_SEED, SPLIT_SEED, FIXED_EVIDENCE_SEED = 42017, 42117, 200042
# CPU execution controls. Inter-op is intentionally conservative for Colab/shared hosts.
CPU_INTRA_OP_THREADS = max(1, min(8, (os.cpu_count() or 2)))
CPU_INTER_OP_THREADS = 2
AUTO_DOWNLOAD_ZIP = True

try:
    tf.config.threading.set_intra_op_parallelism_threads(CPU_INTRA_OP_THREADS)
    tf.config.threading.set_inter_op_parallelism_threads(CPU_INTER_OP_THREADS)
except Exception:
    pass

RUN_SEEDS = [42, 142, 242, 342, 442]
MODEL_INIT_SEEDS = [10042, 10142, 10242, 10342, 10442]
RANDOMK_SEEDS = [911337, 911437, 911537, 911637, 911737]
MIN_CLIENT_SAMPLES, MIN_CLASS_SAMPLES_FOR_DQ = 100, 25

# Revised TADP policy — same policy as Experiment A.
GOOD_CUT, HIGH_CUT = 3.0, 3.5
ADEQUATE_FACTOR_SCORE, WAC_MIN_BASE, ZERO_SCORE_SAFEGUARD = 3.0, 0.70, True
HPS_WEIGHTS = {"dim1":0.25, "dim2":0.15, "dim3":0.10, "dim4":0.10, "dim5":0.30, "dim6":0.10}

# Optional: point this to the FINAL Experiment-A T17 CSV to reuse the exact documentary
# evidence matrix. CIFAR DQ is always recomputed from CIFAR TRAIN-only data.
EXPERIMENT_A_DOCUMENTARY_MATRIX_CSV: Optional[str] = None

SCENARIOS = [
    ("Full FedAvg", "fedavg"),
    ("TADP-VR", "tadp-vr"),
    ("Random-K", "random-k"),
]

DOC_FACTOR_NAMES = {
    "dim1": ["source_reputation", "data_controller", "data_objective"],
    "dim3": ["data_dictionary", "version_logs", "collection_protocol", "definition_updates"],
    "dim4": ["data_freshness", "scheduled_refresh", "retention_clarity"],
    "dim5": ["regulation_coverage", "consent_ethics", "geo_restrictions", "sensitivity_classification", "audits"],
    "dim6": ["license_terms", "ethical_reviews", "redistribution", "user_agreements"],
}
DQ_FACTOR_NAMES = ["completeness", "duplication_rate", "error_rate", "type_consistency",
                   "label_integrity", "feature_distribution_consistency", "feature_support_coverage"]

# ======================================================================================
# 2. UTILITIES
# ======================================================================================

def now_iso(): return datetime.now(timezone.utc).isoformat()

def set_seed(seed):
    seed = int(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    try: tf.keras.utils.set_random_seed(seed)
    except Exception: tf.random.set_seed(seed)
    try: tf.config.experimental.enable_op_determinism()
    except Exception: pass
    return seed

def stable_seed(*parts):
    raw = "|".join(map(str, parts)).encode(); return int(hashlib.sha256(raw).hexdigest()[:8], 16)

def sha256_bytes(*arrays):
    h = hashlib.sha256()
    for a in arrays:
        a = np.ascontiguousarray(a); h.update(str(a.shape).encode()); h.update(str(a.dtype).encode()); h.update(a.tobytes())
    return h.hexdigest()

def weights_sha256(weights): return sha256_bytes(*[np.asarray(w) for w in weights])
def canonical_json_sha256(obj): return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def mean_sd(x):
    v = pd.to_numeric(pd.Series(x), errors="coerce").dropna().to_numpy(float)
    return (float(v.mean()) if len(v) else np.nan, float(v.std(ddof=1)) if len(v) > 1 else np.nan, len(v))

def mean_ci95(x):
    v = pd.to_numeric(pd.Series(x), errors="coerce").dropna().to_numpy(float); n = len(v)
    if not n: return np.nan, np.nan, np.nan, np.nan, 0
    m = float(v.mean())
    if n == 1: return m, np.nan, np.nan, np.nan, 1
    sd = float(v.std(ddof=1)); margin = float(student_t.ppf(0.975, n-1) * sd / np.sqrt(n))
    return m, sd, m-margin, m+margin, n

# ======================================================================================
# 3. CIFAR-10 LOAD + TRAIN-ONLY NON-IID PARTITION
# ======================================================================================

def load_cifar10_official():
    try: (xtr, ytr), (xte, yte) = tf.keras.datasets.cifar10.load_data()
    except Exception as exc:
        raise RuntimeError("CIFAR-10 unavailable. Synthetic fallback is disabled for publication runs.") from exc
    xtr, xte = np.asarray(xtr), np.asarray(xte)
    ytr, yte = np.asarray(ytr).reshape(-1).astype(np.int64), np.asarray(yte).reshape(-1).astype(np.int64)
    if xtr.shape != (50000,32,32,3) or xte.shape != (10000,32,32,3):
        raise RuntimeError(f"Unexpected CIFAR-10 shapes: TRAIN={xtr.shape}, TEST={xte.shape}")
    return (xtr, ytr), (xte, yte)

def split_official_train_for_validation(x, y):
    """Create a stratified validation set exclusively from official CIFAR-10 TRAIN."""
    idx = np.arange(len(y), dtype=np.int64)
    tr_idx, va_idx = train_test_split(
        idx,
        test_size=VALIDATION_FRACTION,
        random_state=SPLIT_SEED,
        stratify=y,
    )
    return (x[tr_idx].copy(), y[tr_idx].copy()), (x[va_idx].copy(), y[va_idx].copy()), tr_idx, va_idx

def dirichlet_partition_indices(y, seed=PARTITION_SEED, max_tries=1000):
    ids = list("ABCDEFGHIJ")
    for attempt in range(max_tries):
        rng = np.random.default_rng(seed + attempt); buckets = {c: [] for c in ids}
        for cls in range(10):
            idx = np.flatnonzero(y == cls); rng.shuffle(idx)
            p = rng.dirichlet(np.full(NUM_CLIENTS, DIRICHLET_ALPHA)); cuts = (np.cumsum(p)[:-1] * len(idx)).astype(int)
            for cid, part in zip(ids, np.split(idx, cuts)): buckets[cid].extend(part.tolist())
        if min(map(len, buckets.values())) >= MIN_CLIENT_SAMPLES:
            return ids, {c: np.asarray(sorted(v), np.int64) for c, v in buckets.items()}, seed + attempt
    raise RuntimeError("Could not construct the predeclared minimum-size TRAIN-only Dirichlet partition.")

def build_raw_clients(x, y, index_map): return {c:(x[idx].copy(), y[idx].copy()) for c, idx in index_map.items()}

def partition_audit(raw_clients):
    rows = []
    for cid, (_, y) in raw_clients.items():
        cnt = np.bincount(y, minlength=10); row = {"client":cid, "n_samples":len(y)}
        row.update({f"class_{k}_count":int(cnt[k]) for k in range(10)})
        row.update({f"class_{k}_pct":100*cnt[k]/max(1,len(y)) for k in range(10)}); rows.append(row)
    return pd.DataFrame(rows)

# ======================================================================================
# 4. LEAKAGE-SAFE TRAIN-ONLY NORMALIZATION
# ======================================================================================

def local_channel_stats(x, chunk=2048):
    n = 0; s = np.zeros(3,np.float64); ss = np.zeros(3,np.float64)
    for start in range(0, len(x), chunk):
        z = x[start:start+chunk].astype(np.float64) / 255.0
        n += z.shape[0]*z.shape[1]*z.shape[2]; s += z.sum((0,1,2)); ss += np.square(z).sum((0,1,2))
    return n, s, ss

def fit_train_only_normalizer(raw_clients):
    tn = 0; ts = np.zeros(3,np.float64); tss = np.zeros(3,np.float64); rows = []
    for cid, (x, _) in raw_clients.items():
        n, s, ss = local_channel_stats(x); tn += n; ts += s; tss += ss
        rows.append({"client":cid, "pixel_count_per_channel":n,
                     **{f"sum_ch{k}":s[k] for k in range(3)}, **{f"sumsq_ch{k}":ss[k] for k in range(3)}})
    mean = ts/tn; var = np.maximum(tss/tn - mean**2, 1e-12); std = np.sqrt(var)
    return mean.astype(np.float32), std.astype(np.float32), pd.DataFrame(rows)

def transform_images(x, mean, std):
    z = x.astype(np.float32)/255.0
    return ((z - mean.reshape(1,1,1,3)) / std.reshape(1,1,1,3)).astype(np.float32)

def preprocess_clients(raw_clients, x_val_raw, x_test_raw, mean, std):
    clients = {c:(transform_images(x,mean,std), y.astype(np.int64)) for c,(x,y) in raw_clients.items()}
    return clients, transform_images(x_val_raw,mean,std), transform_images(x_test_raw,mean,std)

# ======================================================================================
# 5. CIFAR TRAIN-ONLY DQ — CLASS-CONDITIONAL LOW-LEVEL FEATURE DISTRIBUTIONS
# ======================================================================================
# Class-conditional comparison is deliberate: it measures feature consistency within a
# class while avoiding direct or indirect penalization of the intentionally non-IID class
# proportions. The held-out TEST is never used.
# ======================================================================================

MEAN_EDGES, STD_EDGES = np.linspace(0,256,65), np.linspace(0,128,65)
FEATURE_KEYS = [f"mean_ch{k}" for k in range(3)] + [f"std_ch{k}" for k in range(3)]

def empty_hist(): return {k:np.zeros(64,dtype=np.int64) for k in FEATURE_KEYS}

def image_feature_histograms_by_class(x, y, chunk=1024):
    hist = {cls:empty_hist() for cls in range(10)}; counts = np.zeros(10,dtype=np.int64); degenerate = 0
    for start in range(0,len(x),chunk):
        z = x[start:start+chunk].astype(np.float32); yy = y[start:start+chunk]
        means, stds = z.mean((1,2)), z.std((1,2)); degenerate += int((stds.mean(axis=1) <= 1e-6).sum())
        for cls in np.unique(yy):
            m = yy == cls; counts[int(cls)] += int(m.sum())
            for ch in range(3):
                hist[int(cls)][f"mean_ch{ch}"] += np.histogram(means[m,ch], bins=MEAN_EDGES)[0]
                hist[int(cls)][f"std_ch{ch}"] += np.histogram(np.clip(stds[m,ch],0,127.999999), bins=STD_EDGES)[0]
    return hist, counts, degenerate/max(1,len(x))

def exact_duplicate_rate(x):
    seen=set(); dup=0
    for img in x:
        key=hashlib.blake2b(np.ascontiguousarray(img).tobytes(),digest_size=12).digest()
        if key in seen: dup += 1
        else: seen.add(key)
    return dup/max(1,len(x))

def jsd(a,b,eps=1e-12):
    p,q=np.asarray(a,float),np.asarray(b,float); p/=max(eps,p.sum()); q/=max(eps,q.sum()); m=.5*(p+q)
    return float(.5*np.sum(np.where(p>0,p*np.log((p+eps)/(m+eps)),0)) + .5*np.sum(np.where(q>0,q*np.log((q+eps)/(m+eps)),0)))

def support_coverage(local, ref):
    r = np.asarray(ref)>0
    return float((((np.asarray(local)>0)&r).sum()/max(1,r.sum())))

def score_bad_rate(x):
    return 5.0 if x<=.001 else 4.0 if x<=.01 else 3.0 if x<=.02 else 2.0 if x<=.05 else 1.0 if x<=.10 else 0.0

def score_duplicate_rate(x):
    return 5.0 if x<=.01 else 4.0 if x<=.02 else 3.0 if x<=.05 else 2.0 if x<=.10 else 1.0 if x<=.20 else 0.0

def score_jsd(x):
    return 5.0 if x<=.010 else 4.0 if x<=.025 else 3.0 if x<=.050 else 2.0 if x<=.100 else 1.0 if x<=.200 else 0.0

def score_support(x):
    return 5.0 if x>=.900 else 4.0 if x>=.825 else 3.0 if x>=.750 else 2.0 if x>=.650 else 1.0 if x>=.500 else 0.0

def compute_tadp_dq(raw_clients):
    local = {}; global_hist = {cls:empty_hist() for cls in range(10)}; global_counts = np.zeros(10,dtype=np.int64)

    for cid,(x,y) in raw_clients.items():
        finite=np.isfinite(x); missing=1-float(finite.mean()); pixel_invalid=1-float((finite&(x>=0)&(x<=255)).mean())
        labels=np.asarray(y); valid_labels=np.isfinite(labels)&(labels==np.floor(labels))&(labels>=0)&(labels<=9)
        schema_ok=x.ndim==4 and x.shape[1:]==(32,32,3) and np.issubdtype(x.dtype,np.number)
        type_ok=schema_ok and labels.ndim==1 and np.issubdtype(labels.dtype,np.integer)
        h, counts, degenerate=image_feature_histograms_by_class(x,labels)
        for cls in range(10):
            global_counts[cls]+=counts[cls]
            for key in FEATURE_KEYS: global_hist[cls][key]+=h[cls][key]
        local[cid]={"n_samples":len(y),"missing_rate":missing,"pixel_invalid_rate":pixel_invalid,
                    "label_invalid_rate":1-float(valid_labels.mean()),"schema_ok":schema_ok,"type_ok":type_ok,
                    "duplicate_rate":exact_duplicate_rate(x),"degenerate_rate":degenerate,"hist":h,"class_counts":counts}

    rows, factor_map = [], {}
    for cid,d in local.items():
        jsds, coverages = [], []
        for cls in range(10):
            if d["class_counts"][cls] < MIN_CLASS_SAMPLES_FOR_DQ: continue
            ref_n = global_counts[cls]-d["class_counts"][cls]
            if ref_n < MIN_CLASS_SAMPLES_FOR_DQ: continue
            for key in FEATURE_KEYS:
                ref = global_hist[cls][key]-d["hist"][cls][key]
                if ref.sum() <= 0 or d["hist"][cls][key].sum() <= 0: continue
                jsds.append(jsd(d["hist"][cls][key],ref)); coverages.append(support_coverage(d["hist"][cls][key],ref))

        # If no conditional comparison is statistically usable, record minimum-adequate
        # neutral values and expose the zero evaluable-pair count for audit.
        max_jsd = max(jsds) if jsds else .050
        coverage = float(np.mean(coverages)) if coverages else .750
        error_rate = max(d["pixel_invalid_rate"],d["degenerate_rate"])
        factors = {
            "completeness":score_bad_rate(d["missing_rate"]), "duplication_rate":score_duplicate_rate(d["duplicate_rate"]),
            "error_rate":score_bad_rate(error_rate), "type_consistency":5.0 if d["type_ok"] else 0.0,
            "label_integrity":score_bad_rate(d["label_invalid_rate"]), "feature_distribution_consistency":score_jsd(max_jsd),
            "feature_support_coverage":score_support(coverage),
        }
        factor_map[cid]=factors
        rows.append({"client":cid,**factors,"dq_mean":np.mean(list(factors.values())),"max_class_conditional_feature_jsd":max_jsd,
                     "class_conditional_support_coverage_pct":100*coverage,"evaluable_class_feature_pairs":len(jsds),
                     "missing_rate":d["missing_rate"],"duplicate_rate_observed":d["duplicate_rate"],
                     "error_rate_observed":error_rate,"label_invalid_rate":d["label_invalid_rate"],"n_samples":d["n_samples"]})
    return pd.DataFrame(rows), factor_map

# ======================================================================================
# 6. FIXED DOCUMENTARY EVIDENCE — CONTROLLED AND OUTCOME-INDEPENDENT
# ======================================================================================

def _blank_doc(): return {d:{n:np.nan for n in names} for d,names in DOC_FACTOR_NAMES.items()}

def _load_documentary_from_t17(path, client_ids):
    q=pd.read_csv(path); q["client"]=q["client"].astype(str).str.strip().str.upper(); out={}
    for cid in client_ids:
        r=q[q.client.eq(cid)]
        if r.empty: raise RuntimeError(f"Documentary matrix missing client {cid}")
        r=r.iloc[-1]; d=_blank_doc()
        for dim,names in DOC_FACTOR_NAMES.items():
            for name in names:
                col=next((c for c in (f"{dim}_{name}",name) if c in q.columns),None)
                if col is None: raise RuntimeError(f"Missing documentary factor {dim}.{name} in {path}")
                d[dim][name]=float(r[col])
        out[cid]=d
    return out,{c:"IMPORTED" for c in client_ids},"loaded_exact_experiment_A_documentary_matrix"

def _generate_fixed_documentary_matrix(client_ids,seed=FIXED_EVIDENCE_SEED):
    # Controlled composition only: not naturally observed CIFAR metadata and never tuned
    # using model outcomes. Six adequate bundles + four insufficient bundles.
    rng=np.random.default_rng(seed); bundles=[]
    for k in range(6):
        d={dim:{name:float(rng.integers(3,6)) for name in names} for dim,names in DOC_FACTOR_NAMES.items()}; bundles.append((f"ADEQ{k+1}",d))
    forced=[("dim1","data_controller"),("dim3","version_logs"),("dim5","geo_restrictions"),("dim6","user_agreements")]
    for k,(zdim,zname) in enumerate(forced):
        d={dim:{name:float(rng.integers(2,5)) for name in names} for dim,names in DOC_FACTOR_NAMES.items()}; d[zdim][zname]=0.0
        bundles.append((f"INS{k+1}",d))
    order=rng.permutation(len(bundles)); out={}; labels={}
    for cid,idx in zip(sorted(client_ids),order): labels[cid],out[cid]=bundles[int(idx)]
    return out,labels,"fixed_controlled_6_adequate_4_insufficient_subfactor_matrix"

def prepare_documentary_evidence(client_ids):
    if EXPERIMENT_A_DOCUMENTARY_MATRIX_CSV:
        doc,labels,mode=_load_documentary_from_t17(EXPERIMENT_A_DOCUMENTARY_MATRIX_CSV,client_ids)
    else:
        doc,labels,mode=_generate_fixed_documentary_matrix(client_ids)
    rows=[]
    for cid in sorted(client_ids):
        row={"client":cid,"evidence_bundle":labels[cid],"assignment_mode":mode,"evidence_seed":FIXED_EVIDENCE_SEED}
        for dim,vals in doc[cid].items():
            for name,value in vals.items(): row[f"{dim}_{name}"]=value
        rows.append(row)
    table=pd.DataFrame(rows); matrix_hash=canonical_json_sha256(table.to_dict("records")); table["matrix_sha256"]=matrix_hash
    return doc,table,matrix_hash,mode

# ======================================================================================
# 7. REVISED TADP: HPS + WAC + ZERO-SCORE SAFEGUARD
# ======================================================================================

def dimension_means(factors): return {d:float(np.mean(list(v.values()))) if v else 0.0 for d,v in factors.items()}
def hps_score(dims): return float(sum(HPS_WEIGHTS[d]*dims[d] for d in HPS_WEIGHTS))

def weighted_adequacy_coverage(factors):
    coverage=0.0
    for dim,w in HPS_WEIGHTS.items():
        vals=[float(v) for v in factors[dim].values()]
        coverage += w*(sum(v>=ADEQUATE_FACTOR_SCORE for v in vals)/max(1,len(vals)))
    return float(coverage/sum(HPS_WEIGHTS.values()))

def zero_factors(factors):
    return [f"{dim}.{name}" for dim,vals in factors.items() for name,v in vals.items() if np.isfinite(v) and float(v)<=0]

def apply_tadp_policy(hps,wac,zeros):
    if ZERO_SCORE_SAFEGUARD and zeros:
        return "ZERO_SCORE_SAFEGUARD","QUARANTINE","QUARANTINED_ZERO_FACTOR","factor score 0: "+", ".join(zeros)
    if hps<GOOD_CUT: return "QUARANTINE","QUARANTINE","QUARANTINED_LOW_HPS",f"HPS {hps:.3f} < {GOOD_CUT:.3f}"
    if hps<HIGH_CUT:
        ok=wac>=WAC_MIN_BASE
        return "REVIEW",("ACCEPT" if ok else "REJECT"),("ACCEPTED_AFTER_REVIEW" if ok else "NOT_ADMITTED_LOW_ADEQUACY"),f"Review WAC {wac:.3f}"
    ok=wac>=WAC_MIN_BASE
    return ("AUTO_ACCEPT" if ok else "ADEQUACY_CHECK"),("ACCEPT" if ok else "REJECT"),("ACCEPTED" if ok else "NOT_ADMITTED_LOW_ADEQUACY"),f"Direct band WAC {wac:.3f}"

def build_tadp_governance(client_ids,doc,dq_map):
    rows=[]
    for cid in sorted(client_ids):
        factors={dim:dict(vals) for dim,vals in doc[cid].items()}; factors["dim2"]=dict(dq_map[cid])
        factors={d:factors[d] for d in ["dim1","dim2","dim3","dim4","dim5","dim6"]}
        dims=dimension_means(factors); hps=hps_score(dims); wac=weighted_adequacy_coverage(factors); zeros=zero_factors(factors)
        initial,final,status,reason=apply_tadp_policy(hps,wac,zeros)
        row={"client":cid,"HPS":hps,"WAC":wac,"zero_score_count":len(zeros),"zero_factors":";".join(zeros),
             "initial_action":initial,"final_action":final,"status":status,"reason":reason}
        row.update({f"{d}_mean":dims[d] for d in dims})
        for dim,vals in factors.items():
            for name,v in vals.items(): row[f"{dim}_{name}"]=v
        rows.append(row)
    table=pd.DataFrame(rows); accepted=sorted(table.loc[table.final_action.eq("ACCEPT"),"client"].tolist())
    return table,accepted

# ======================================================================================
# 8. MODEL + FEDERATED TRAINING
# ======================================================================================

def build_model():
    # No BatchNorm: avoids aggregation ambiguity for non-trainable moving statistics.
    return keras.Sequential([
        layers.Input((32,32,3)),
        layers.Conv2D(32,3,padding="same",activation="relu"), layers.Conv2D(32,3,padding="same",activation="relu"),
        layers.MaxPooling2D(), layers.Dropout(.20),
        layers.Conv2D(64,3,padding="same",activation="relu"), layers.Conv2D(64,3,padding="same",activation="relu"),
        layers.MaxPooling2D(), layers.Dropout(.30),
        layers.Conv2D(128,3,padding="same",activation="relu"), layers.GlobalAveragePooling2D(),
        layers.Dense(128,activation="relu"), layers.Dropout(.40), layers.Dense(10,activation="softmax")
    ],name="cifar10_fl_cnn")

def evaluate(model,x_test,y_test):
    p=model.predict(x_test,batch_size=256,verbose=0); pred=np.argmax(p,axis=1); onehot=tf.keras.utils.to_categorical(y_test,10)
    try: auc=float(roc_auc_score(onehot,p,multi_class="ovr",average="macro"))
    except Exception: auc=np.nan
    return {"accuracy":float(accuracy_score(y_test,pred)),
            "precision":float(precision_score(y_test,pred,average="macro",zero_division=0)),
            "recall":float(recall_score(y_test,pred,average="macro",zero_division=0)),
            "f1":float(f1_score(y_test,pred,average="macro",zero_division=0)),"roc_auc":auc}

def natural_steps(n): return int(math.ceil(n/BATCH_SIZE)*LOCAL_EPOCHS)

def batch_sequence(n,steps,seed):
    rng=np.random.default_rng(seed); perm=rng.permutation(n); cursor=0
    for _ in range(int(steps)):
        if cursor>=n: perm=rng.permutation(n); cursor=0
        end=min(cursor+BATCH_SIZE,n); idx=perm[cursor:end]; cursor=end
        if not len(idx): perm=rng.permutation(n); cursor=min(BATCH_SIZE,n); idx=perm[:cursor]
        yield idx

def _reset_adam_state(optimizer):
    """Reset per-client Adam state while preserving the configured learning rate."""
    for v in optimizer.variables:
        name = v.name.lower()
        if "learning_rate" in name:
            continue
        try:
            v.assign(tf.zeros_like(v))
        except Exception:
            pass


def make_compiled_local_trainer():
    """Build one reusable local model/optimizer and one compiled train step per scenario."""
    model = build_model()
    optimizer = keras.optimizers.Adam(LR_LOCAL)
    loss_fn = keras.losses.SparseCategoricalCrossentropy()

    # Build optimizer slots once so they can be reset cheaply between clients.
    zero_grads = [tf.zeros_like(v) for v in model.trainable_variables]
    optimizer.apply_gradients(zip(zero_grads, model.trainable_variables))
    _reset_adam_state(optimizer)

    @tf.function(reduce_retracing=True)
    def train_step(xb, yb):
        with tf.GradientTape() as tape:
            p = model(xb, training=True)
            loss = loss_fn(yb, p)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    return model, optimizer, train_step


def train_exact_steps_reuse(model, optimizer, train_step, X, y, steps, seed):
    if steps<=0 or len(y)==0: return 0
    _reset_adam_state(optimizer)
    for idx in batch_sequence(len(y),steps,seed):
        xb=tf.convert_to_tensor(X[idx], dtype=tf.float32)
        yb=tf.convert_to_tensor(y[idx], dtype=tf.int64)
        train_step(xb,yb)
    return int(steps)


def weighted_average_weights(weight_sets,sample_sizes):
    w=np.asarray(sample_sizes,float); w/=w.sum()
    return [np.sum(np.stack([ws[j] for ws in weight_sets])*w.reshape((-1,)+(1,)*weight_sets[0][j].ndim),axis=0)
            for j in range(len(weight_sets[0]))]

def fixed_round_step_budget(cohort,clients): return sum(natural_steps(len(clients[c][1])) for c in cohort)

def allocate_exact_steps(total_steps,selected,clients):
    sizes=np.array([len(clients[c][1]) for c in selected],float)
    if total_steps<len(selected): raise RuntimeError("Matched optimizer-step budget is smaller than Random-K cohort size.")
    raw=total_steps*sizes/sizes.sum(); alloc=np.maximum(np.floor(raw).astype(int),1)
    while alloc.sum()>total_steps:
        candidates=np.where(alloc>1)[0]; idx=candidates[np.argmin(raw[candidates]-alloc[candidates])]; alloc[idx]-=1
    rem=total_steps-alloc.sum(); order=np.argsort(-(raw-np.floor(raw)))
    for k in range(rem): alloc[order[k%len(order)]]+=1
    if alloc.sum()!=total_steps: raise RuntimeError("Exact Random-K optimizer-step allocation failed.")
    return {c:int(s) for c,s in zip(selected,alloc)}

def select_random_k(run_idx,all_clients,k):
    # FIXED random cohort for the full run: same cohort persistence as TADP-VR.
    rng=np.random.default_rng(RANDOMK_SEEDS[run_idx]); return sorted(rng.choice(sorted(all_clients),size=k,replace=False).tolist())

def run_federated(scenario,run_idx,initial_weights,clients,selected,x_val,y_val,x_test,y_test,
                  step_rows,round_metric_rows,forced_steps_per_round=None):
    if not selected:
        return {"status":"INFEASIBLE_EMPTY_COHORT","selected_ids":"","n_clients":0,"optimizer_steps":0,"client_rounds":0,
                **{m:np.nan for m in ["accuracy","precision","recall","f1","roc_auc"]},"time_s":0.0}

    set_seed(stable_seed("scenario",RUN_SEEDS[run_idx],scenario))
    global_model=build_model(); global_model.set_weights([w.copy() for w in initial_weights])
    init_hash=weights_sha256(global_model.get_weights()); total_steps=0; t0=time.perf_counter()

    # Reuse a single compiled local trainer across every client and round in this scenario.
    local_model, local_optimizer, local_train_step = make_compiled_local_trainer()

    for rnd in range(1,NUM_ROUNDS_FL+1):
        allocation = ({c:natural_steps(len(clients[c][1])) for c in selected} if forced_steps_per_round is None
                      else allocate_exact_steps(forced_steps_per_round,selected,clients))
        global_weights=[np.array(w,copy=True) for w in global_model.get_weights()]
        local_weights=[]; sizes=[]; round_steps=0

        for cid in selected:
            local_model.set_weights([w.copy() for w in global_weights])
            steps=allocation[cid]
            train_exact_steps_reuse(
                local_model,local_optimizer,local_train_step,
                clients[cid][0],clients[cid][1],steps,
                stable_seed("local",run_idx,rnd,cid)
            )
            local_weights.append([np.array(w,copy=True) for w in local_model.get_weights()])
            sizes.append(len(clients[cid][1])); round_steps+=steps

        global_model.set_weights(weighted_average_weights(local_weights,sizes)); total_steps+=round_steps
        step_rows.append({"run":run_idx,"run_number":run_idx+1,"scenario":scenario,"round":rnd,
                          "selected_clients":len(selected),"selected_ids":";".join(selected),
                          "optimizer_steps":round_steps,"initial_weights_sha256":init_hash})

        # TRAIN-derived validation only. Official TEST remains untouched until training is complete.
        vm=evaluate(global_model,x_val,y_val)
        round_metric_rows.append({"run":run_idx,"run_number":run_idx+1,"scenario":scenario,"round":rnd,
                                  "selected_clients":len(selected),"optimizer_steps_cumulative":total_steps,
                                  **{f"val_{k}":v for k,v in vm.items()}})
        print(f"   round {rnd:02d}/{NUM_ROUNDS_FL} | val Acc={vm['accuracy']:.4f} | "
              f"F1={vm['f1']:.4f} | AUC={vm['roc_auc']:.4f}")

    # One official TEST evaluation after the complete predeclared training budget.
    metrics=evaluate(global_model,x_test,y_test); elapsed=time.perf_counter()-t0
    del local_model, global_model
    tf.keras.backend.clear_session(); gc.collect()
    return {"status":"OK","selected_ids":";".join(selected),"n_clients":len(selected),
            "optimizer_steps":total_steps,"client_rounds":len(selected)*NUM_ROUNDS_FL,**metrics,"time_s":elapsed}

# ======================================================================================
# 9. ANALYSIS TABLES
# ======================================================================================

def performance_summary(perf):
    metrics=["accuracy","precision","recall","f1","roc_auc","time_s","optimizer_steps","client_rounds"]
    rows=[]
    for sc,q in perf.groupby("scenario",sort=False):
        row={"scenario":sc,"n_runs":q.run.nunique(),"clients_per_round":q.n_clients.mean()}
        for m in metrics:
            mu,sd,n=mean_sd(q[m]); row[f"{m}_mean"],row[f"{m}_sd"],row[f"{m}_n"]=mu,sd,n
        rows.append(row)
    return pd.DataFrame(rows)

def matched_control_audit(step_df,init_df,random_cohorts,vr_cohort):
    rows=[]
    for run in range(N_RUNS):
        vr=step_df[(step_df.run==run)&(step_df.scenario=="TADP-VR")].sort_values("round")
        rk=step_df[(step_df.run==run)&(step_df.scenario=="Random-K")].sort_values("round")
        hv=init_df[(init_df.run==run)&(init_df.scenario=="TADP-VR")].initial_weights_sha256
        hr=init_df[(init_df.run==run)&(init_df.scenario=="Random-K")].initial_weights_sha256
        same_rounds=list(vr.round)==list(rk.round) and len(vr)==NUM_ROUNDS_FL
        same_k=same_rounds and np.array_equal(vr.selected_clients.to_numpy(),rk.selected_clients.to_numpy())
        same_steps=same_rounds and np.array_equal(vr.optimizer_steps.to_numpy(),rk.optimizer_steps.to_numpy())
        same_init=not hv.empty and not hr.empty and hv.iloc[-1]==hr.iloc[-1]
        rk_ids=tuple(random_cohorts[run]); fixed_random=not rk.empty and len(set(rk.selected_ids.astype(str)))==1
        rows.append({"run":run,"run_number":run+1,"K":len(vr_cohort),"vr_clients":";".join(vr_cohort),
                     "random_k_clients":";".join(rk_ids),"same_rounds":same_rounds,"same_K_per_round":same_k,
                     "same_optimizer_steps_per_round":same_steps,"same_initial_weights":same_init,
                     "random_k_fixed_across_rounds":fixed_random,
                     "all_matched":same_rounds and same_k and same_steps and same_init and fixed_random})
    return pd.DataFrame(rows)

def paired_utility_analysis(perf):
    # Paired by run. Positive method-minus-full means higher utility than Full FedAvg.
    wide=perf.pivot(index="run",columns="scenario",values=["accuracy","f1","roc_auc"])
    detail=[]; summary=[]
    for method in ["TADP-VR","Random-K"]:
        for metric in ["accuracy","f1","roc_auc"]:
            d=(wide[(metric,method)]-wide[(metric,"Full FedAvg")]).dropna()*100.0
            for run,val in d.items(): detail.append({"run":int(run),"run_number":int(run)+1,"method":method,"metric":metric,"delta_vs_full_pp":float(val),"absolute_gap_to_full_pp":abs(float(val))})
            m,sd,lo,hi,n=mean_ci95(d)
            summary.append({"method":method,"metric":metric,"n_paired_runs":n,"mean_delta_vs_full_pp":m,"sd_delta_pp":sd,"ci95_low_pp":lo,"ci95_high_pp":hi,
                            "mean_absolute_gap_to_full_pp":float(np.mean(np.abs(d))) if len(d) else np.nan})

    # Direct governance-selection control: TADP-VR - Random-K.
    for metric in ["accuracy","f1","roc_auc"]:
        d=(wide[(metric,"TADP-VR")]-wide[(metric,"Random-K")]).dropna()*100.0
        m,sd,lo,hi,n=mean_ci95(d)
        summary.append({"method":"TADP-VR minus Random-K","metric":metric,"n_paired_runs":n,
                        "mean_delta_vs_full_pp":m,"sd_delta_pp":sd,"ci95_low_pp":lo,"ci95_high_pp":hi,
                        "mean_absolute_gap_to_full_pp":np.nan})
    return pd.DataFrame(detail),pd.DataFrame(summary)

def validation_stabilization(round_metrics):
    """Reviewer-facing descriptive diagnostic; never used to choose TEST-facing settings."""
    rows=[]
    for (run,scenario),q in round_metrics.groupby(["run","scenario"],sort=False):
        q=q.sort_values("round")
        acc=q["val_accuracy"].to_numpy(float)
        f1=q["val_f1"].to_numpy(float)
        window=min(3,len(q))
        acc_gain=float(acc[-1]-acc[-window]) if window>=2 else np.nan
        f1_gain=float(f1[-1]-f1[-window]) if window>=2 else np.nan
        rows.append({"run":int(run),"run_number":int(run)+1,"scenario":scenario,
                     "rounds_observed":len(q),"last_window":window,
                     "val_accuracy_gain_last_window":acc_gain,
                     "val_f1_gain_last_window":f1_gain,
                     "descriptive_stabilized":bool(np.isfinite(acc_gain) and np.isfinite(f1_gain) and abs(acc_gain)<0.01 and abs(f1_gain)<0.01)})
    return pd.DataFrame(rows)

# ======================================================================================
# 10. PUBLICATION FIGURE — PAIRED UTILITY GAP TO FULL FEDAVG
# ======================================================================================

def make_utility_figure(detail):
    import matplotlib as mpl, matplotlib.pyplot as plt
    mpl.rcParams.update({"font.family":"serif","font.serif":["STIXGeneral","DejaVu Serif"],"mathtext.fontset":"stix",
                         "font.size":8.5,"axes.labelsize":9,"xtick.labelsize":8,"ytick.labelsize":8,
                         "axes.spines.top":False,"axes.spines.right":False,"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none"})
    labels={"accuracy":"Accuracy","f1":"Macro-F1","roc_auc":"ROC-AUC"}; metrics=["accuracy","f1","roc_auc"]
    fig,ax=plt.subplots(figsize=(5.2,3.15)); y=[]; ypos=[]; yt=[]; pos=0
    markers={"TADP-VR":"D","Random-K":"o"}

    for metric in metrics:
        for method in ["TADP-VR","Random-K"]:
            vals=detail[(detail.metric==metric)&(detail.method==method)].delta_vs_full_pp.to_numpy(float)
            m,sd,lo,hi,n=mean_ci95(vals); offset=-.12 if method=="TADP-VR" else .12
            jitter=np.linspace(-.045,.045,len(vals)) if len(vals)>1 else np.zeros(len(vals))
            ax.scatter(vals,np.full(len(vals),pos+offset)+jitter,s=18,facecolor="white",edgecolor="0.4",linewidth=.6,zorder=3)
            if np.isfinite(m):
                xerr=np.array([[m-lo],[hi-m]]) if np.isfinite(lo) and np.isfinite(hi) else None
                ax.errorbar(m,pos+offset,xerr=xerr,fmt=markers[method],ms=5,color="black",capsize=2.5,lw=1.0,zorder=5)
        ypos.append(pos); yt.append(labels[metric]); pos+=1

    ax.axvline(0,color="0.35",ls="--",lw=.8); ax.set_yticks(ypos,yt); ax.invert_yaxis()
    ax.set_xlabel("Paired difference from Full FedAvg (percentage points)")
    ax.grid(axis="x",lw=.35,alpha=.20)
    ax.text(.01,1.03,"← lower than Full FedAvg",transform=ax.transAxes,ha="left",fontsize=7.2,color="0.35")
    ax.text(.99,1.03,"higher than Full FedAvg →",transform=ax.transAxes,ha="right",fontsize=7.2,color="0.35")
    # legend proxies
    ax.plot([],[],"D",color="black",ms=5,label="TADP-VR"); ax.plot([],[],"o",color="black",ms=5,label="Random-$K$")
    ax.legend(frameon=False,ncol=2,loc="lower right")
    fig.tight_layout(pad=.5)

    stem=FIG_DIR/"Fig_CIFAR10_Paired_Utility_vs_Full"
    fig.savefig(Path(str(stem)+"_600dpi.png"),dpi=600,bbox_inches="tight",pad_inches=.03,facecolor="white")
    fig.savefig(Path(str(stem)+".pdf"),bbox_inches="tight",pad_inches=.03,facecolor="white")
    fig.savefig(Path(str(stem)+".svg"),bbox_inches="tight",pad_inches=.03,facecolor="white")
    plt.show()

# ======================================================================================
# 11. MAIN EXPERIMENT
# ======================================================================================

def main():
    print("="*110); print(EXPERIMENT_VERSION); print("="*110)
    print(f"TensorFlow {tf.__version__} | GPU(s): {len(tf.config.list_physical_devices('GPU'))} | Python {platform.python_version()}")
    print(f"Execution target: CPU-optimized | intra-op={CPU_INTRA_OP_THREADS} | inter-op={CPU_INTER_OP_THREADS}")
    print(f"Training budget: {NUM_ROUNDS_FL} FL rounds × {LOCAL_EPOCHS} local epoch(s) | batch={BATCH_SIZE}")

    # ---------- Official TEST isolated; validation is carved only from official TRAIN ----------
    (x_official_train,y_official_train),(x_test_raw,y_test)=load_cifar10_official()
    test_hash=sha256_bytes(x_test_raw,y_test)
    (x_train_raw,y_train_raw),(x_val_raw,y_val),train_idx,val_idx = split_official_train_for_validation(x_official_train,y_official_train)
    split_hash=sha256_bytes(train_idx,val_idx)

    cids,index_map,partition_seed=dirichlet_partition_indices(y_train_raw)
    raw_clients=build_raw_clients(x_train_raw,y_train_raw,index_map)
    part=partition_audit(raw_clients); part["partition_seed"]=partition_seed; part["alpha"]=DIRICHLET_ALPHA
    part.to_csv(TABLE_DIR/"client_partition.csv",index=False)

    # ---------- Federated-TRAIN-only preprocessing ----------
    mean,std,local_stats=fit_train_only_normalizer(raw_clients)
    clients,x_val,x_test=preprocess_clients(raw_clients,x_val_raw,x_test_raw,mean,std)
    local_stats.to_csv(TABLE_DIR/"preprocessing_local_sufficient_stats.csv",index=False)
    pd.DataFrame([{"dataset":"CIFAR-10","official_train_n":len(x_official_train),
                   "federated_train_n":len(x_train_raw),"validation_n":len(x_val_raw),"heldout_test_n":len(x_test_raw),
                   "validation_fraction":VALIDATION_FRACTION,"split_seed":SPLIT_SEED,"split_sha256":split_hash,
                   "clients":NUM_CLIENTS,"dirichlet_alpha":DIRICHLET_ALPHA,"partition_seed":partition_seed,
                   "normalization_source":"aggregated federated-TRAIN-only RGB count/sum/sumsq",
                   "validation_used_for_partition":False,"validation_used_for_governance":False,
                   "test_used_for_partition":False,"test_used_for_preprocessing_fit":False,
                   "test_used_for_governance":False,"test_used_for_training":False,"test_raw_sha256":test_hash,
                   **{f"train_mean_ch{k}":float(mean[k]) for k in range(3)},
                   **{f"train_std_ch{k}":float(std[k]) for k in range(3)}}]).to_csv(TABLE_DIR/"preprocessing_audit.csv",index=False)

    print(f"\n✅ Official TRAIN={x_official_train.shape} -> federated TRAIN={x_train_raw.shape} + VALIDATION={x_val_raw.shape}")
    print(f"   untouched official TEST={x_test_raw.shape}")
    print(f"   TRAIN-only RGB mean={np.round(mean,6).tolist()} | std={np.round(std,6).tolist()}")

    # ---------- Governance computed ONCE and frozen across all five ML runs ----------
    dq_table,dq_map=compute_tadp_dq(raw_clients)
    doc_map,doc_table,doc_hash,evidence_mode=prepare_documentary_evidence(cids)
    tadp_table,vr_cohort=build_tadp_governance(cids,doc_map,dq_map)
    dq_hash=canonical_json_sha256(dq_table.to_dict("records")); dq_table["dq_sha256"]=dq_hash
    dq_table.to_csv(TABLE_DIR/"tadp_dq_7factor_cifar10.csv",index=False)
    doc_table.to_csv(TABLE_DIR/"documentary_evidence_matrix.csv",index=False)
    tadp_table["documentary_matrix_sha256"]=doc_hash; tadp_table["dq_sha256"]=dq_hash; tadp_table["evidence_assignment_mode"]=evidence_mode
    tadp_table.to_csv(TABLE_DIR/"tadp_governance_decisions.csv",index=False)
    pd.DataFrame([{
        "governance_computed_once_before_training":True, "governance_frozen_across_runs":True,
        "documentary_evidence_mode":evidence_mode, "documentary_matrix_sha256":doc_hash, "cifar_dq_sha256":dq_hash,
        "TADP_VR_K":len(vr_cohort), "TADP_VR_clients":";".join(vr_cohort),
        "test_used_for_governance":False, "class_proportions_used_for_admission":False
    }]).to_csv(TABLE_DIR/"governance_pretraining_audit.csv",index=False)

    if not vr_cohort:
        raise RuntimeError("TADP-VR admitted zero CIFAR clients. Fail closed; do not fabricate a training cohort.")

    print("\n🔒 Governance frozen BEFORE model training")
    print(f"   TADP-VR cohort: {len(vr_cohort)}/{NUM_CLIENTS} -> {vr_cohort}")
    print(f"   Documentary matrix SHA-256: {doc_hash}")
    print(f"   CIFAR DQ SHA-256: {dq_hash}")

    # TADP-VR natural per-round step budget; Random-K must match it exactly.
    vr_steps_per_round=fixed_round_step_budget(vr_cohort,clients)
    perf_rows=[]; step_rows=[]; round_metric_rows=[]; init_rows=[]; random_cohorts={}

    for run_idx in range(N_RUNS):
        print("\n"+"="*110); print(f"RUN {run_idx+1}/{N_RUNS}"); print("="*110)

        # Common W0 across the 3 scenarios in this run.
        set_seed(MODEL_INIT_SEEDS[run_idx]); m0=build_model(); initial_weights=[np.array(w,copy=True) for w in m0.get_weights()]
        init_hash=weights_sha256(initial_weights); del m0; tf.keras.backend.clear_session(); gc.collect()

        random_cohort=select_random_k(run_idx,cids,len(vr_cohort)); random_cohorts[run_idx]=random_cohort
        cohorts={"Full FedAvg":sorted(cids),"TADP-VR":vr_cohort,"Random-K":random_cohort}

        for scenario,_ in SCENARIOS:
            selected=cohorts[scenario]; forced=vr_steps_per_round if scenario=="Random-K" else None
            init_rows.append({"run":run_idx,"run_number":run_idx+1,"scenario":scenario,"run_seed":RUN_SEEDS[run_idx],
                              "model_init_seed":MODEL_INIT_SEEDS[run_idx],"initial_weights_sha256":init_hash})
            result=run_federated(scenario,run_idx,initial_weights,clients,selected,x_val,y_val,x_test,y_test,
                                 step_rows,round_metric_rows,forced_steps_per_round=forced)
            perf_rows.append({"run":run_idx,"run_number":run_idx+1,"scenario":scenario,"rounds":NUM_ROUNDS_FL,
                              "local_epochs":LOCAL_EPOCHS,"batch_size":BATCH_SIZE,"alpha":DIRICHLET_ALPHA,
                              "initial_weights_sha256":init_hash,"governance_frozen":True,
                              "test_used_for_training_or_governance":False,**result,"timestamp":now_iso()})
            print(f"▶ {scenario:10s} | K={result['n_clients']:2d} | steps={result['optimizer_steps']:4d} | "
                  f"Acc={result['accuracy']:.4f} | F1={result['f1']:.4f} | AUC={result['roc_auc']:.4f}")

        pd.DataFrame(perf_rows).to_csv(OUT/"performance_metrics.csv",index=False)
        pd.DataFrame(step_rows).to_csv(LEDGER_DIR/"step_parity_rounds.csv",index=False)
        pd.DataFrame(round_metric_rows).to_csv(OUT/"validation_learning_curves.csv",index=False)
        pd.DataFrame(init_rows).to_csv(LEDGER_DIR/"initialization_audit.csv",index=False)

    # ---------- Audits + paired utility analysis ----------
    perf=pd.DataFrame(perf_rows); step_df=pd.DataFrame(step_rows); round_metrics=pd.DataFrame(round_metric_rows); init_df=pd.DataFrame(init_rows)
    summary=performance_summary(perf); summary.to_csv(TABLE_DIR/"performance_mean_sd.csv",index=False)
    stabilization=validation_stabilization(round_metrics); stabilization.to_csv(TABLE_DIR/"validation_stabilization.csv",index=False)
    matched=matched_control_audit(step_df,init_df,random_cohorts,vr_cohort); matched.to_csv(TABLE_DIR/"matched_control.csv",index=False)
    if len(matched)!=N_RUNS or not matched.all_matched.all():
        raise RuntimeError("TADP-VR ↔ Random-K matching failed. Inspect tables/matched_control.csv")

    detail,utility=paired_utility_analysis(perf)
    detail.to_csv(TABLE_DIR/"paired_utility_vs_full_by_run.csv",index=False)
    utility.to_csv(TABLE_DIR/"paired_utility_summary.csv",index=False)

    # Exact common-initialization check for all three scenarios in each run.
    init_check=(init_df.groupby("run").initial_weights_sha256.nunique()==1)
    if not init_check.all(): raise RuntimeError("Common model initialization failed within at least one run.")

    # Held-out validation / TEST integrity audits.
    if sha256_bytes(x_test_raw,y_test)!=test_hash: raise RuntimeError("Held-out TEST raw arrays changed during execution.")
    if sha256_bytes(train_idx,val_idx)!=split_hash: raise RuntimeError("TRAIN/validation split indices changed during execution.")

    make_utility_figure(detail)

    # ---------- Configuration + reproducibility package ----------
    config={"experiment_version":EXPERIMENT_VERSION,"scenarios":[s for s,_ in SCENARIOS],"num_clients":NUM_CLIENTS,
            "n_runs":N_RUNS,"dirichlet_alpha":DIRICHLET_ALPHA,"partition_seed":partition_seed,"rounds":NUM_ROUNDS_FL,
            "local_epochs":LOCAL_EPOCHS,"batch_size":BATCH_SIZE,"learning_rate":LR_LOCAL,
            "good_cut":GOOD_CUT,"high_cut":HIGH_CUT,"wac_min":WAC_MIN_BASE,"adequate_factor_score":ADEQUATE_FACTOR_SCORE,
            "zero_score_safeguard":ZERO_SCORE_SAFEGUARD,"hps_weights":HPS_WEIGHTS,
            "documentary_evidence_mode":evidence_mode,"documentary_evidence_seed":FIXED_EVIDENCE_SEED,
            "documentary_matrix_sha256":doc_hash,"cifar_dq_sha256":dq_hash,"heldout_test_sha256":test_hash,
            "run_seeds":RUN_SEEDS,"model_init_seeds":MODEL_INIT_SEEDS,"randomk_seeds":RANDOMK_SEEDS,
            "random_k_design":"one fixed random K-client cohort per run, reused across all rounds; exact VR step budget per round"}
    (OUT/"experiment_config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")

    manifest=[]
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p.name not in {"SHA256_MANIFEST.csv","TADP_CIFAR10_3SCENARIO_Q1_v3_RESULTS.zip"}:
            manifest.append({"file":str(p.relative_to(OUT)),"bytes":p.stat().st_size,"sha256":hashlib.sha256(p.read_bytes()).hexdigest()})
    pd.DataFrame(manifest).to_csv(OUT/"SHA256_MANIFEST.csv",index=False)

    zip_path=OUT/"TADP_CIFAR10_3SCENARIO_Q1_v3_RESULTS.zip"
    with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=9) as zf:
        for p in sorted(OUT.rglob("*")):
            if p.is_file() and p!=zip_path: zf.write(p,arcname=str(p.relative_to(OUT)))

    print("\n"+"="*110); print("✅ CIFAR-10 3-SCENARIO GENERALIZATION COMPLETE"); print("="*110)
    print(f"TADP-VR frozen cohort: {len(vr_cohort)}/{NUM_CLIENTS} {vr_cohort}")
    print("Random-K matched control: PASS")
    print("Common initialization: PASS")
    print("Held-out TEST leakage audit: PASS")
    print(f"Outputs: {OUT}\nZIP: {zip_path}")
    print("\nPerformance summary:"); print(summary.to_string(index=False))
    print("\nPaired utility summary (percentage points):"); print(utility.to_string(index=False))
    print("\nValidation stabilization diagnostic:"); print(stabilization.to_string(index=False))

    if AUTO_DOWNLOAD_ZIP and str(ROOT)=="/content":
        try:
            from google.colab import files as colab_files
            print("\n⬇️ Starting automatic ZIP download...")
            colab_files.download(str(zip_path))
        except Exception as exc:
            print(f"Automatic download skipped: {exc}")

    return {"performance":perf,"summary":summary,"utility":utility,"tadp":tadp_table,"dq":dq_table,
            "matched_control":matched,"validation_curves":round_metrics,"stabilization":stabilization,
            "output_dir":OUT,"zip_path":zip_path}

if __name__ == "__main__":
    results = main()


TADP-CIFAR10 v3.0 — CPU-optimized 3-scenario generalization
TensorFlow 2.20.0 | GPU(s): 0 | Python 3.13.15
Execution target: CPU-optimized | intra-op=2 | inter-op=2
Training budget: 10 FL rounds × 1 local epoch(s) | batch=128

✅ Official TRAIN=(50000, 32, 32, 3) -> federated TRAIN=(45000, 32, 32, 3) + VALIDATION=(5000, 32, 32, 3)
   untouched official TEST=(10000, 32, 32, 3)
   TRAIN-only RGB mean=[0.49113500118255615, 0.48173999786376953, 0.4460490047931671] | std=[0.24720799922943115, 0.24354399740695953, 0.26155999302864075]

🔒 Governance frozen BEFORE model training
   TADP-VR cohort: 6/10 -> ['A', 'D', 'E', 'F', 'I', 'J']
   Documentary matrix SHA-256: 0b03863b4f85cf7a7fbbfb205f088749f22a8b01ea6325604fa60e4f88b875e3
   CIFAR DQ SHA-256: 98289fe3bf9fb73e0ab718574fb06708fd37b57cd755d1dcc7e77505cfdf2aea

RUN 1/5
   round 01/10 | val Acc=0.1182 | F1=0.0454 | AUC=0.6926
   round 02/10 | val Acc=0.1826 | F1=0.0859 | AUC=0.7248
   round 03/10 | val Acc=0.2248 | F1=0.1590 | AUC=0.7692
   

TypeError: 'method' object is not iterable